#### Data Extraction and selection

In [1]:
import pandas as pd
import random
import os
import glob

In [2]:
folder_path = '../CSV-03-11/03-11/' 
all_files = glob.glob(os.path.join(folder_path, "*.csv"))
all_files

['../CSV-03-11/03-11\\LDAP.csv',
 '../CSV-03-11/03-11\\MSSQL.csv',
 '../CSV-03-11/03-11\\NetBIOS.csv',
 '../CSV-03-11/03-11\\Portmap.csv',
 '../CSV-03-11/03-11\\Syn.csv',
 '../CSV-03-11/03-11\\UDP.csv',
 '../CSV-03-11/03-11\\UDPLag.csv']

In [3]:
sampled_dataframes = []
TARGET_ROWS_PER_FILE = 20000

print("Starting memory-efficient random sampling...")

for file in all_files:
    file_name = os.path.basename(file)
    print(f"Processing {file_name}...")
    
    # Step A: Quickly count total rows in the file (without loading it into memory)
    with open(file, 'r', encoding='utf-8') as f:
        total_rows = sum(1 for _ in f)
    
    # Step B: Calculate the probability needed to get roughly 20,000 rows
    keep_probability = TARGET_ROWS_PER_FILE / total_rows
    
    # Step C: Load the CSV, randomly skipping rows based on our probability
    df_sample = pd.read_csv(
        file, 
        skiprows=lambda x: x > 0 and random.random() > keep_probability,
        low_memory=False
    )
    
    # Clean up column names (strip annoying spaces)
    df_sample.columns = df_sample.columns.str.strip()
    sampled_dataframes.append(df_sample)
    print(f" -> Kept {len(df_sample)} random rows from {file_name}")

Starting memory-efficient random sampling...
Processing LDAP.csv...
 -> Kept 20016 random rows from LDAP.csv
Processing MSSQL.csv...
 -> Kept 19938 random rows from MSSQL.csv
Processing NetBIOS.csv...
 -> Kept 19924 random rows from NetBIOS.csv
Processing Portmap.csv...
 -> Kept 20190 random rows from Portmap.csv
Processing Syn.csv...
 -> Kept 19853 random rows from Syn.csv
Processing UDP.csv...
 -> Kept 20009 random rows from UDP.csv
Processing UDPLag.csv...
 -> Kept 19970 random rows from UDPLag.csv


In [4]:
# 2. Merge all the randomly sampled chunks into one dataset
print("Merging datasets...")
main_train_dataset = pd.concat(sampled_dataframes, ignore_index=True)

# Save your beautiful, balanced, randomly sampled training data!
main_train_dataset.to_csv("CICDDOS_Test_Data.csv", index=False)
print(f"Done! Final Testing dataset size: {len(main_train_dataset)} rows.")

Merging datasets...
Done! Final Testing dataset size: 139900 rows.
